# 分页调试：逐页拉取 + 记录每页数据

输入 BV 号，逐页拉取一级评论，记录每页的 cursor、all_count、实际评论数、楼中楼数。
用于排查爬取数量对不上的根因。

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

BV_ID = "BV1rYCvYJEWQ"   # ← 改成你自己的 BV 号
print(f"目标视频: {BV_ID}")

In [ ]:
import requests
from bilibili_api import video as bv_video

# 获取 oid
v = bv_video.Video(bvid=BV_ID)
info = await v.get_info()
oid = info["aid"]
title = info["title"]
print(f"标题: {title}")
print(f"oid:  {oid}")
print(f"API 评论总数: {info.get('stat', {}).get('reply', '?')}")

In [ ]:
import asyncio
import time

API = "https://api.bilibili.com/x/v2/reply/main"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Referer": f"https://www.bilibili.com/video/{BV_ID}",
}

log_rows = []  # 每页的记录

def fetch_page(cursor: int):
    params = {"oid": oid, "type": 1, "mode": 2, "ps": 20}
    if cursor > 0:
        params["next"] = cursor
    r = requests.get(API, params=params, headers=HEADERS, timeout=15)
    data = r.json()
    if data.get("code") != 0:
        raise RuntimeError(f"API 错误: code={data['code']}, msg={data.get('message')}")
    return data["data"]

next_cursor = 0
page = 0
total_top = 0
total_replies = 0
t0 = time.perf_counter()

while True:
    page += 1
    resp = await asyncio.to_thread(fetch_page, next_cursor)
    
    cursor = resp.get("cursor") or {}
    all_count = cursor.get("all_count", 0)
    next_cursor_new = cursor.get("next", 0)
    
    replies = resp.get("replies") or []
    reply_counts = [r.get("rcount", 0) for r in replies]
    this_top = len(replies)
    this_replies = sum(reply_counts)
    
    total_top += this_top
    total_replies += this_replies
    
    log_rows.append({
        "page": page,
        "cursor_in": next_cursor,
        "cursor_out": next_cursor_new,
        "all_count": all_count,
        "top_count": this_top,
        "reply_count": this_replies,
        "cum_top": total_top,
        "cum_total": total_top + total_replies,
    })
    
    print(f"第{page:3d}页 | cursor {next_cursor:>10d} → {next_cursor_new:>10d} | "
          f"all_count={all_count:>5d} | 本页:{this_top:>2d}条一级 + {this_replies:>3d}条楼中楼 | "
          f"累计:{total_top}+{total_replies}={total_top+total_replies}")
    
    if next_cursor_new == 0:
        print(f"\n✅ next=0，翻页结束")
        break
    
    next_cursor = next_cursor_new
    await asyncio.sleep(0.3)

elapsed = time.perf_counter() - t0
print(f"\n总结: {page} 页, 一级 {total_top} + 楼中楼 {total_replies} = 总计 {total_top+total_replies}")
print(f"API 报告的 all_count: {log_rows[0]['all_count'] if log_rows else '?'}")
print(f"耗时: {elapsed:.1f}s")

## 数据分析 — 各页明细

In [ ]:
print(f"{'页码':>5} {'cursor_in':>12} {'cursor_out':>12} {'all_count':>10} {'本页一级':>8} {'楼中楼':>8} {'累计一级':>8} {'累计总计':>8}")
print("-" * 85)
for r in log_rows:
    print(f"{r['page']:>5} {r['cursor_in']:>12} {r['cursor_out']:>12} {r['all_count']:>10} {r['top_count']:>8} {r['reply_count']:>8} {r['cum_top']:>8} {r['cum_total']:>8}")

## 关键对比

In [ ]:
first_all_count = log_rows[0]["all_count"] if log_rows else 0
last_all_count = log_rows[-1]["all_count"] if log_rows else 0
final_cum = log_rows[-1]["cum_total"] if log_rows else 0

print(f"API 报告的 all_count (首页): {first_all_count}")
print(f"API 报告的 all_count (末页): {last_all_count}")
print(f"爬虫实际爬到: {final_cum} (一级{log_rows[-1]['cum_top']} + 楼中楼{final_cum - log_rows[-1]['cum_top']})")
print(f"差值: {first_all_count - final_cum}")

if first_all_count != last_all_count:
    print(f"\n⚠️ all_count 在翻页过程中发生了变化: {first_all_count} → {last_all_count}")

if final_cum < first_all_count:
    pct = final_cum / first_all_count * 100
    print(f"\n⚡ 爬取覆盖率: {pct:.1f}%，缺失 {100-pct:.1f}%")
    print("   可能原因:")
    print("   1. 缺失的评论已被删除/隐藏 → API 不计入返回")
    print("   2. 楼中楼分页未完全拉取（rcount 不准）")
    print("   3. API 翻页提前终止")
    print("\n   建议: 手工打开视频评论区，翻到最后几页，")
    print("   对照本表确认是否有评论缺失。")

## 验证楼中楼分页

抽查某一页的第一条有楼中楼的评论，看看 rcount 和实际拉取数量是否一致。

In [ ]:
# 找第一条有较多楼中楼的评论
import random

TARGET_PAGE = 1  # 抽查第几页

resp = await asyncio.to_thread(fetch_page, 0)
replies = resp.get("replies") or []

# 找 rcount > 5 的评论
targets = [(r, r["rcount"]) for r in replies if r.get("rcount", 0) > 5]

if targets:
    target, rcount = targets[0]
    rpid = target["rpid"]
    print(f"目标: rpid={rpid}, rcount={rcount}, 内容: {target.get('content',{}).get('message','')[:50]}…")
    
    from bilibili_api import comment as bili_comment
    c = bili_comment.Comment(oid=oid, type_=bili_comment.CommentResourceType.VIDEO, rpid=rpid)
    
    all_sub = []
    for pi in range(1, 20):  # 最多 20 页
        sub_resp = await c.get_sub_comments(page_index=pi, page_size=20)
        sub_replies = sub_resp.get("replies") or []
        if not sub_replies:
            break
        all_sub.extend(sub_replies)
        page_info = sub_resp.get("page") or {}
        total_sub = page_info.get("count", 0)
        print(f"  楼中楼第{pi}页: 返回{len(sub_replies)}条, API总数={total_sub}, 累计={len(all_sub)}")
        if len(all_sub) >= total_sub:
            break
        await asyncio.sleep(0.3)
    
    print(f"\n  结果: rcount={rcount}, 实际拉到={len(all_sub)}, 匹配={'✅' if len(all_sub) == rcount else '❌ 差'+str(abs(rcount-len(all_sub)))}")
else:
    print("首页没有 rcount>5 的评论，换个视频试试")